# NeuroGolf 2026 — Safe Blend

## The simplest possible blend that actually works.

**Strategy:**
1. Find all `submission.zip` from attached notebooks
2. For each ONNX model: try loading it **raw, zero modification**
3. If it works and has `input`/`output` names → keep **original bytes unchanged**
4. If it needs rename → fix it **correctly** (capture old names FIRST, then rename ALL references)
5. Keep cheapest model per task
6. Identity fallback for missing tasks

**Why previous blends broke:**
- `onnx.load_from_string()` + `SerializeToString()` can silently corrupt valid models
- Even when no changes are needed, re-serialization can break custom ops, metadata, etc.
- The old rename bug (rename first, capture old names after) broke intermediate node references

**This notebook:** never touches a model that already works. Period.

In [ ]:
!pip install -q onnxruntime onnx


In [ ]:
import os, zipfile, re, io, csv, glob, time, traceback
import numpy as np
import onnxruntime as ort
import onnx
from pathlib import Path
from collections import Counter, defaultdict

ort.set_default_logger_severity(3)
print("Ready.")


In [ ]:
SOURCE_ZIPS = []
for zp in sorted(glob.glob('/kaggle/input/notebooks/**/submission.zip', recursive=True)):
    SOURCE_ZIPS.append((zp, Path(zp).parent.name))

print(f'Found {len(SOURCE_ZIPS)} sources:')
for zp, name in SOURCE_ZIPS:
    print(f'  {name}  ({os.path.getsize(zp)/1024/1024:.1f} MB)')


In [ ]:
def safe_load_model(raw_bytes, task_id=None):
    """
    Минимальный фикс: только приводим input/output к 'input'/'output'.
    Никаких инференс-тестов. Доверяем исходникам.
    Возвращает (success, model_bytes, cost)
    """
    try:
        model = onnx.load_model_from_string(raw_bytes)
        g = model.graph
        if not g.input or not g.output:
            return False, raw_bytes, float('inf')

        rename_map = {}
        old_in = g.input[0].name
        old_out = g.output[0].name
        if old_in != 'input':  rename_map[old_in] = 'input'
        if old_out != 'output': rename_map[old_out] = 'output'

        # Применяем переименование если нужно
        if rename_map:
            g.input[0].name = rename_map.get(old_in, old_in)
            g.output[0].name = rename_map.get(old_out, old_out)
            
            for node in g.node:
                node.input[:]  = [rename_map.get(n, n) for n in node.input]
                node.output[:] = [rename_map.get(n, n) for n in node.output]
                
            for vi in g.value_info:
                vi.name = rename_map.get(vi.name, vi.name)
            for init in g.initializer:
                init.name = rename_map.get(init.name, init.name)

        model.ir_version = 8
        fixed_bytes = model.SerializeToString()
        
        # Кост: кол-во параметров + размер файла
        params = sum(int(np.prod(i.dims)) for i in g.initializer if i.dims)
        cost = params + len(fixed_bytes)
        return True, fixed_bytes, cost

    except Exception:
        # Если вообще не парсится, пропускаем
        return False, raw_bytes, float('inf')

print('safe_load_model simplified.')


def _fix_subgraph(sg, rename_map):
    for node in sg.node:
        for i in range(len(node.input)):
            if node.input[i] in rename_map:
                node.input[i] = rename_map[node.input[i]]
        for i in range(len(node.output)):
            if node.output[i] in rename_map:
                node.output[i] = rename_map[node.output[i]]
        for attr in node.attribute:
            if attr.HasField('g'):
                _fix_subgraph(attr.g, rename_map)
            for s in attr.graphs:
                _fix_subgraph(s, rename_map)
    for vi in list(sg.input) + list(sg.output) + list(sg.value_info):
        if vi.name in rename_map:
            vi.name = rename_map[vi.name]

print('safe_load_model defined.')


In [ ]:
best_models = {}
model_costs = {}
source_tracker = {}
src_stats = defaultdict(lambda: {'ok':0,'fail':0,'won':0,'total':0})

# ==========================================================
# --- НАСТРОЙКИ ТОП-РЕШЕНИЯ ---
# ==========================================================
TOP_SOLUTION_PATH = "/kaggle/input/notebooks/konbu17/neurogolf-2026-blended-401-tasks-lb-5344/submission.zip"
# Эти таски берем ТОЛЬКО из топа и больше ни с чем не сравниваем
IMMUTABLE_TASKS = {
    #'task101.onnx', 
    #'task133.onnx',
}

locked_tasks = set() 
# ==========================================================

t0 = time.time()

# --- ЭТАП 1: Захват ПРИОРИТЕТНЫХ моделей (только IMMUTABLE) ---
print(f"--- Stage 1: Locking Immutable Tasks ---")
if os.path.exists(TOP_SOLUTION_PATH):
    label = "TOP_PRIORITY"
    try:
        with zipfile.ZipFile(TOP_SOLUTION_PATH) as zf:
            for entry in zf.namelist():
                if not entry.endswith('.onnx'): continue
                basename = os.path.basename(entry)
                
                if basename in IMMUTABLE_TASKS:
                    raw = zf.read(entry)
                    ok, data, cost = safe_load_model(raw, basename)
                    if ok:
                        best_models[basename] = data
                        model_costs[basename] = cost
                        source_tracker[basename] = f"LOCKED:{label}"
                        locked_tasks.add(basename)
                        src_stats[label]['won'] += 1
                        src_stats[label]['ok'] += 1
        print(f"  Locked {len(locked_tasks)} tasks.")
    except Exception as e:
        print(f"  Error locking tasks: {e}")

# Добавим ТОП-сабмит в общий список для сравнения остальных тасок, 
# если его там еще нет (чтобы не прописывать вручную в SOURCE_ZIPS)
all_sources = SOURCE_ZIPS.copy()
if TOP_SOLUTION_PATH not in [x[0] for x in all_sources]:
    all_sources.append((TOP_SOLUTION_PATH, "TOP_FOR_BLENDING"))

# --- ЭТАП 2: Общий блендинг (сравнение по стоимости) ---
print(f"\n--- Stage 2: Competitive Blending (by cost) ---")
for zip_path, label in all_sources:
    if not os.path.exists(zip_path): continue
    
    print(f'Processing {label}...')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            for entry in zf.namelist():
                if not entry.endswith('.onnx'): continue
                m = re.search(r'task(\d{3})\.onnx', os.path.basename(entry))
                if not m: continue
                tid = f'task{m.group(1)}.onnx'
                
                # Если таска заблокирована на Этапе 1 — не трогаем её
                if tid in locked_tasks:
                    continue
                
                raw = zf.read(entry)
                ok, data, cost = safe_load_model(raw, tid)
                
                if not ok:
                    src_stats[label]['fail'] += 1
                    continue
                
                src_stats[label]['total'] += 1
                src_stats[label]['ok'] += 1
                
                # Сравниваем: если из топа пришла таска дешевле, чем из другого места — берем её
                if tid not in best_models or cost < model_costs[tid]:
                    best_models[tid] = data
                    model_costs[tid] = cost
                    source_tracker[tid] = label
                    src_stats[label]['won'] += 1
    except Exception as e:
        print(f'  ERROR in {label}: {e}')

print(f'\n{"="*60}')
print(f'Blend done: {len(best_models)} models.')
print(f'Locked: {len(locked_tasks)} | Competitive: {len(best_models)-len(locked_tasks)}')
print(f'{"="*60}')

for src, cnt in Counter(source_tracker.values()).most_common():
    print(f'  {src}: {cnt}')

In [ ]:
def make_fallback():
    """Minimal valid ONNX: Conv 1x1 identity. Always works."""
    w = np.eye(10, dtype=np.float32).reshape(10,10,1,1)
    b = np.zeros(10, dtype=np.float32)
    node = onnx.helper.make_node('Conv', ['input','w','b'], ['output'],
                                 kernel_shape=[1,1], pads=[0,0,0,0])
    g = onnx.helper.make_graph([node], 'fb',
        [onnx.helper.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1,10,30,30])],
        [onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1,10,30,30])],
        [onnx.helper.make_tensor('w', onnx.TensorProto.FLOAT, [10,10,1,1], w.flatten()),
         onnx.helper.make_tensor('b', onnx.TensorProto.FLOAT, [10], b)])
    m = onnx.helper.make_model(g, opset_imports=[onnx.helper.make_opsetid('',12)])
    m.ir_version = 8
    return m.SerializeToString()

fb = make_fallback()
for i in range(400):
    tid = f'task{i:03d}.onnx'
    if tid not in best_models:
        best_models[tid] = fb
        model_costs[tid] = 10**8
        source_tracker[tid] = 'fallback'

print(f'Total models: {len(best_models)}/400')


In [ ]:
# Write submission.zip
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in sorted(best_models):
        zf.writestr(n, best_models[n])

with open('submission.zip','wb') as f:
    f.write(buf.getvalue())

# Write submission.csv
with open('submission.csv','w',newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_id','total_cost'])
    for n in sorted(best_models):
        w.writerow([n.replace('.onnx',''), model_costs[n]])

sz = buf.tell()
print(f'\n{"="*60}')
print(f'submission.zip: {sz/1024:.1f} KB, {len(best_models)} models')
print(f'Press Submit!')
print(f'{"="*60}')
